# RAG workshop: an AI that looks things up

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hugobb/IndigenousAI/blob/main/notebooks/rag_workshop_beginner.ipynb)

An AI chatbot answers from what it learned during its training. RAG (Retrieval-Augmented Generation) lets it look things up first. RAG finds the passages of a document that match your question, and the AI writes its answer from them.

In this notebook you try each step of RAG on one document: the programme of this workshop. You don't need to know how to code. The grey boxes are cells. To run a cell, click the ▶️ button on its left. Go from top to bottom.

## 0. Setup (5 min)

**Do this:**
1. In the menu at the top, click **Runtime**, then **Change runtime type**. Choose **T4 GPU** and click **Save**.
2. Click ▶️ on the **Setup** cell below.
3. If a message says "This notebook was not authored by Google", click **Run anyway**.
4. Wait until the cell shows **✅ Ready**. This takes 1 to 2 minutes.

If the cell shows ⚠️ instead, everything still works, only slower. Tell an assistant.

If a cell later shows an error that ends with "is not defined", run the Setup cell again, then that cell.

In [ ]:
#@title Setup: click ▶️ and wait for ✅ Ready (1 to 2 minutes) { display-mode: "form" }
# You don't need to read this cell. It installs what is missing, loads the two AI models and
# defines the helpers the workshop uses. The interesting ones (ask, similarity, search, split,
# ask_with_documents) are also shown in each section, in a collapsed "Code" cell.
import logging
import warnings

# Hide three harmless notices: tqdm's progress bars and a macOS note from torch (outside Colab), and
# Colab's note that no Hugging Face token is set (the two models are public and need none).
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", message=r"\s*The secret `HF_TOKEN` does not exist in your Colab secrets")
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    %pip install -q sentence-transformers transformers accelerate

import os
import re
from collections.abc import Iterator
from dataclasses import dataclass
from queue import Empty
from threading import Thread
import numpy as np


# ======================================================================
# content.py
# ======================================================================
"""Everything the notebook works with: the programme, the example sentences and the example questions.

The document is the programme of this workshop (Future Imaginaries of Indigenous AI, Abundant
Intelligences, November 19 and 20, 2026), with its schedule written as prose and short speaker bios,
saved in documents/workshop_programme.txt. The example values were picked with find_demo_values.py.
tests/test_effects.py (embeddings and search) and tests/test_llm_effects.py (the two Qwen models) check
that each one does what the notebook text says.
"""


def _load(name: str) -> str:
    """Read a file from documents/. When the notebook is built, each call is replaced by the text itself."""
    from pathlib import Path

    return (Path(__file__).resolve().parent / "documents" / name).read_text(encoding="utf-8")


# ---------------------------------------------------------------- the programme
_HEADER, _BODY = 'Title: Future Imaginaries of Indigenous AI, workshop programme (Abundant Intelligences)\nSource: https://abundant-intelligences.net/ka-hawaii-pae-aina-pod/\nRetrieved: 2026-11-20\nLicense: programme by the workshop organisers; lab, pod and speaker facts from abundant-intelligences.net (team, AIAI Lab and Ka Hawaiʻi Pae ʻĀina Pod pages), the speakers\' own pages, and Wikipedia "Solomon Enos" (oldid 1379642239, CC BY-SA 4.0)\nShortened: schedule rewritten as prose, speaker bios added.\n\nFuture Imaginaries of Indigenous AI is a two-day workshop by Abundant Intelligences, held on November 19 and 20, 2026, at the Create(x) Labs of the University of Hawaiʻi – West Oʻahu. It is hosted by the Ka Hawaiʻi Pae ʻĀina Pod. The workshop has two goals: to imagine machines of the future that fit the needs and hopes of participants\' communities, through a guided creation exercise, and to give a basic, hands-on introduction to machine learning.\n\n== About Abundant Intelligences ==\nAbundant Intelligences develops AI methods grounded in Indigenous epistemologies, aimed at abundance rather than scarcity. The program works along three axes. The Integrations Axis brings Indigenous knowledge practices together with the knowledge frameworks used by AI researchers. The Imaginaries Axis develops future imaginaries and speculative designs of Indigenous AI. The Intelligences Axis brings Indigenous perspectives to technical challenges in AI research. In short: we listen to each other, we dream our futures together, and we build those visions of the future.\n\n== The Future Imaginaries workshops ==\nThe Future Imaginaries of Indigenous AI workshops belong to the Imaginaries Axis. They help communities, practitioners and critics form their own vocabularies and visions of the futures they want, without falling into existing ideas of how these technologies should develop.\n\n== Day 1: Thursday, November 19 ==\nThe first day runs from 1:00 pm to 4:45 pm in room G-129 of Create(x). It opens at 1:00 pm with a 15-minute welcome: Kari Noe, director of the Create(x) Lab, introduces the Hawaiʻi Pod, and Ceyda Yolgörmez introduces the Future Imaginaries of Indigenous AI. At 1:15 pm, Solomon Enos and Kamuela Enos give a 30-minute talk, "Imagining abundant intelligences". After a 15-minute break, Suzanne Kite leads an hour-long imagining exercise at 2:00 pm, "Towards the future of sovereign intelligences". After a second 15-minute break, the imaginaries workshop continues at 3:15 pm for an hour and a half. The day closes at 4:45 pm.\n\n== Day 2: Friday, November 20 ==\nThe second day runs from 9:30 am to 4:00 pm in room D-141. It opens at 9:30 am with a 15-minute welcome. From 9:45 am to 10:30 am, the session "Setting the parameters of future AIs" covers existing philosophies, imaginaries and applications in the Ka Hawaiʻi Pae ʻĀina Pod, with talks by Manulani Meyer, Mele Moniz and Purple Maiʻa. After a 15-minute break, Karim Jerbi and Jason Lewis introduce the AIAI Lab at 10:45 am. Then Hugo Berard and Hamza Abdelhedi give the introduction to machine learning. Together the two sessions run 75 minutes, until noon. Lunch is from 12:00 to 1:00 pm. At 1:00 pm, participants start the 75-minute hands-on session, "Running your own AI". After a 15-minute break, Yorguin José Mantilla Ramos gives a 15-minute talk at 2:30 pm, "Can we trust AI?". The hands-on workshop then continues for 45 minutes, from 2:45 pm to 3:30 pm. At 3:30 pm, Jason Lewis and Kari Noe lead 30 minutes of reflections and future directions, and the workshop closes at 4:00 pm.\n\n== The Ka Hawaiʻi Pae ʻĀina Pod ==\nThe Ka Hawaiʻi Pae ʻĀina Pod is one of the research pods of Abundant Intelligences, based on Oʻahu. It is hosted by the Create(x) Immersive Media and Data Visualization Lab at the University of Hawaiʻi – West Oʻahu, and its community partner is Purple Maiʻa. Its work covers environmental stewardship, language revitalization, creative practice and community-based data stewardship. Kari Noe is the pod director and Anolani Saizon is the pod coordinator.\n\n== The AIAI Lab ==\nThe AIAI Lab is the technical team of Abundant Intelligences. It builds technical capacity for Indigenous-led AI projects: it gives technical advice, helps build prototypes, connects technical and non-technical collaborators, and runs training and events, including curriculum for the Summer Institute for Indigenous AI. The lab follows Indigenous data sovereignty principles, including OCAP and CARE. Its co-directors are Karim Jerbi of the Université de Montréal and Jason Edward Lewis of Concordia University. Hugo Berard is a postdoctoral researcher in the lab, and its student tech team includes Hamza Abdelhedi, Yorguin Mantilla Ramos and Antoine Bellemare. The lab works with Concordia University, the Université de Montréal, Mila, IVADO and UNIQUE.\n\n== Speakers ==\n=== Kari Noe ===\nKari Noe co-directs the Create(x) emerging media lab at the University of Hawaiʻi – West Oʻahu and is a PhD research assistant at the LAVA lab at the University of Hawaiʻi at Mānoa. She works on human-computer interaction, extended reality and game development, especially how emerging media supports learning.\n\n=== Ceyda Yolgörmez ===\nCeyda Yolgörmez is a postdoctoral researcher at Concordia University, at the Indigenous Futures Research Cluster of Abundant Intelligences. Her PhD combined social theory with interactive technologies such as large machine learning models and social robots, and proposes a "sociology of machines".\n\n=== Solomon Enos ===\nSolomon Enos is a Native Hawaiian artist, illustrator and activist, born in Mākaha Valley on Oʻahu. Solomon Enos\'s work has been shown at the National Museum of the American Indian, the Bishop Museum and the Pitt Rivers Museum in Oxford, England. Solomon Enos illustrated the books Akua Hawaiʻi and The Epic Tale of Hiʻiakaikapoliopele, and received a grant from the Joan Mitchell Foundation in 2017.\n\n=== Kamuela Enos ===\nKamuela Enos is a co-investigator of Abundant Intelligences at the University of Hawaiʻi at Mānoa and a member of the Ka Hawaiʻi Pae ʻĀina Pod. He founded the university\'s Office of Indigenous Innovation, which researches and scales traditional practices for Indigenous communities, and was director of social enterprise at MAʻO Organic Farms.\n\n=== Suzanne Kite ===\nSuzanne Kite is an Oglála Lakȟóta performance artist, composer and scholar. She directs the Wihanble S\'a Center for Indigenous AI at Bard College.\n\n=== Manulani Aluli Meyer ===\nManulani Aluli Meyer works on Indigenous epistemology. She holds a doctorate in philosophy of education from Harvard and is Konohiki for Kūlana o Kapolei at the University of Hawaiʻi – West Oʻahu. She wrote Hoʻoulu: Our Time of Becoming.\n\n=== Melemaikalani Moniz ===\nMelemaikalani (Mele) Moniz is a Kanaka ʻŌiwi legal scholar and a postdoctoral researcher at the University of Hawaiʻi at Mānoa. Her work covers Western legal systems, environmental justice and Indigenous rights. She holds an SJD in Indigenous Peoples Law and Policy from the University of Arizona.\n\n=== Jason Edward Lewis ===\nJason Edward Lewis is a digital media theorist, poet and software designer. He is Professor of Computation Arts at Concordia University, holds the University Research Chair in Computational Media and the Indigenous Future Imaginary, and founded the Obx Laboratory for Experimental Media.\n\n=== Karim Jerbi ===\nKarim Jerbi is Professor of Psychology at the Université de Montréal and a member of Mila. He holds a Canada Research Chair in Computational Neuroscience and directs UNIQUE, the Quebec Neuro-AI research centre. He studies brain oscillations and networks, and the meeting points of neuroscience, AI and art.\n\n=== Hugo Berard ===\nHugo Berard is a postdoctoral researcher with Abundant Intelligences, in the AIAI Lab. He holds a PhD in computer science from Mila, where he applied game theory to improve image-generation algorithms. He works on responsible AI built from local initiatives and community needs, and on how AI can support citizen participation in local decisions. He is active on Mila\'s sustainability committee, which works to reduce AI\'s environmental footprint, and facilitates Climate Fresk workshops.\n\n=== Hamza Abdelhedi ===\nHamza Abdelhedi is a PhD student in biomedical engineering at the Université de Montréal, in the CoCo Lab and at Mila, supervised by Karim Jerbi, and a member of the AIAI Lab. He holds an MSc in computer science (artificial intelligence) from the Université de Montréal and an engineering diploma in telecommunications from SUP\'COM in Tunisia. He uses M/EEG recordings and computational models to study brain dynamics and to compare neural networks with human brain activity, and maintains open-source tools such as MNE-Denoise.\n\n=== Yorguin José Mantilla Ramos ===\nYorguin José Mantilla Ramos is an electronics engineer and a member of the AIAI Lab\'s student tech team. He is interested in the interpretability and explainability of models, how data and knowledge are represented, how cognition works, and the philosophy of science.\n\n=== Purple Maiʻa ===\nPurple Maiʻa is a nonprofit based in Honolulu. It teaches and supports culturally grounded technology makers, mainly Native Hawaiian and Pacific Islander learners.\n'.split("\n\n", 1)
SOURCE = dict(line.split(": ", 1) for line in _HEADER.splitlines())  # Title, Source, Retrieved, License, Shortened
PROGRAMME = _BODY.strip()  # the programme without its 5-line attribution block
PROGRAMME_URL = SOURCE["Source"]
PROGRAMME_RETRIEVED = SOURCE["Retrieved"]

# ---------------------------------------------------------------- 1. The problem: the AI alone
WORKSHOP = "the Future Imaginaries of Indigenous AI workshop"
NO_DOC_QUESTION = f"When does the hands-on session start at {WORKSHOP}?"
NO_DOC_ANSWER = "1:00"  # in the AI's answer with the programme, not without it
TEACHERS_QUESTION = f"Who gives the introduction to machine learning at {WORKSHOP}?"
TEACHER = "Berard"
# Section 5 asks again, with the programme, in this short form.
WITH_DOC_QUESTION = "When does the hands-on session start?"
NO_DOC_ANSWER_IN_PROGRAMME = "At 1:00 pm, participants start the 75-minute hands-on session"

# ---------------------------------------------------------------- 2. Embeddings
SAME_MEANING = ("We learn about machine learning.", "Students study how AI is trained.")
SAME_WORD = ("We learn about machine learning.", "The washing machine is broken.")
UNRELATED = ("We learn about machine learning.", "I had soup for lunch.")
LOVE_HATE = ("I love winter.", "I hate winter.")

# ---------------------------------------------------------------- 3. Search, over the sections
N_SECTIONS = 20  # the Introduction, 6 sections and 13 speakers; tests/test_effects.py checks it
HIT_QUESTION = "Who are the co-directors of the AIAI Lab?"
HIT_SECTION = "The AIAI Lab"
# The section that holds the answer is not first: the Introduction, which also mentions "an introduction
# to machine learning", comes before it.
MISS_QUESTION = "Who gives the introduction to machine learning?"
MISS_SECTION = "Day 2: Friday, November 20"
MISS_ANSWER = "Hugo Berard and Hamza Abdelhedi"
NONSENSE_QUESTION = "What is the best pizza topping?"

# ---------------------------------------------------------------- 4. Splitting
DEMO_SIZE = 50  # words per chunk in sections 4 to 6
DEMO_OVERLAP = 10
TINY_SIZE = 5
SPLIT_QUESTION = "Who gives the talk Can we trust AI?"
CUT_PHRASE = "Yorguin José Mantilla Ramos gives a 15-minute talk at 2:30 pm"  # cut at DEMO_SIZE words
CUT_SECOND_PART = "gives a 15-minute talk at 2:30 pm"  # how the next chunk starts
SPLIT_ANSWER = "Yorguin"  # in the AI's answer with DEMO_OVERLAP, not without overlap

# ---------------------------------------------------------------- 5. Answering
TOP1_VS_TOP3_QUESTION = "Which room is used on Thursday?"
TOP1_VS_TOP3_ANSWER = "G-129"  # in the AI's answer with top=3, not with top=1
TOP1_VS_TOP3_ANSWER_IN_PROGRAMME = "in room G-129"  # in the 2nd best chunk, not the 1st
OUT_OF_DOC_QUESTION = "What is the Wi-Fi password?"
OUT_OF_DOC_ABSENT = ["password", "wi-fi", "wifi"]

# ---------------------------------------------------------------- 6. Wrap-up
WRAP_UP_QUESTION = MISS_QUESTION  # the question from sections 1 and 3, now with the whole pipeline
WRAP_UP_ANSWER = "Berard"


# ======================================================================
# toolkit.py
# ======================================================================
"""The workshop's machinery: chunking, similarity, search, prompts and the two AI models."""



EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"


class MiniLMEmbedder:
    """Turns text into embeddings (384 numbers each) with a small, fast model."""

    def __init__(self, device: str):
        from sentence_transformers import SentenceTransformer

        self.model = SentenceTransformer(EMBED_MODEL, device=device)

    def embed(self, texts: list[str]) -> np.ndarray:
        return self.model.encode(list(texts), normalize_embeddings=True, convert_to_numpy=True)


# ---------------------------------------------------------------- the answer-writing model
CHAT_MODEL_GPU = "Qwen/Qwen3-1.7B"
CHAT_MODEL_CPU = "Qwen/Qwen3-0.6B"
MAX_NEW_TOKENS = 200


def model_dtype(device: str, model_name: str):
    """Full precision (float32) on the GPU, where half precision (float16) can overflow with Qwen3 on a T4,
    and for the small CPU model, because bfloat16 can be very slow on CPUs like Colab's. Larger models on
    the CPU use bfloat16, half the memory."""
    import torch

    return torch.float32 if device == "cuda" or model_name == CHAT_MODEL_CPU else torch.bfloat16


def dtype_argument(device: str, model_name: str) -> dict:
    """The precision argument for from_pretrained. transformers 4.56 renamed torch_dtype to dtype:
    older versions crash on dtype=, newer ones print a warning for torch_dtype=."""
    import transformers

    major, minor = (int(part) for part in transformers.__version__.split(".")[:2])
    return {"dtype" if (major, minor) >= (4, 56) else "torch_dtype": model_dtype(device, model_name)}


def stream_worker(target, streamer) -> Iterator[str]:
    """Run `target` (the model writing) in the background and pass on its text as it arrives.
    If it fails, stop waiting and raise instead of hanging forever."""
    failure: list[BaseException] = []

    def work():
        try:
            target()
        except BaseException as error:  # noqa: BLE001 - any failure must reach the participant
            failure.append(error)
            streamer.text_queue.put(streamer.stop_signal)

    worker = Thread(target=work, daemon=True)
    worker.start()
    try:
        yield from streamer
    except Empty:
        raise RuntimeError("The model took too long to answer.") from None
    worker.join(timeout=1)
    if failure:
        raise RuntimeError(f"The model stopped with an error: {failure[0]}") from failure[0]


class QwenGenerator:
    """Writes answers with a small Qwen model. Greedy decoding: the same question gives the same answer."""

    def __init__(self, model_name: str, device: str):
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name, **dtype_argument(device, model_name)).to(device)
        self.model.eval()

    def generate(self, prompt: str) -> Iterator[str]:
        from transformers import TextIteratorStreamer

        text = self.tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
        streamer = TextIteratorStreamer(self.tokenizer, skip_prompt=True, skip_special_tokens=True, timeout=180)
        settings = dict(
            **inputs,
            streamer=streamer,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
        )
        yield from stream_worker(lambda: self.model.generate(**settings), streamer)


class FakeGenerator:
    """A stand-in for tests: repeats the first passage instead of thinking."""

    def generate(self, prompt: str) -> Iterator[str]:
        yield "(test answer) "
        first = next((line for line in prompt.splitlines() if line.startswith("[1] ")), prompt)
        for word in first.removeprefix("[1] ").split():
            yield word + " "


@dataclass
class Models:
    embedder: MiniLMEmbedder
    generator: QwenGenerator | FakeGenerator
    device: str
    llm_name: str


MODELS: Models | None = None
NOT_READY = "Run the Setup cell first."


def load_models() -> Models:
    """Load both models, on the GPU when there is one. The result is also kept in MODELS."""
    global MODELS
    MODELS = _load_models()
    return MODELS


def _load_models() -> Models:
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    embedder = MiniLMEmbedder(device)
    if os.environ.get("RAG_WORKSHOP_FAKE_LLM") == "1":
        return Models(embedder, FakeGenerator(), device, "fake")
    name = os.environ.get("RAG_WORKSHOP_LLM") or (CHAT_MODEL_GPU if device == "cuda" else CHAT_MODEL_CPU)
    return Models(embedder, QwenGenerator(name, device), device, name)


def running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


def status_line(device: str, in_colab: bool | None = None) -> str:
    """The Setup cell's last line. Only Colab has a Runtime menu that can switch on a GPU."""
    if device == "cuda":
        return "✅ Ready. You can go on to section 1."
    if in_colab is None:
        in_colab = running_in_colab()
    slow = "⚠️ Ready, but there is no GPU, so the AI is slower (up to a minute per answer) and makes more mistakes. You can carry on."
    colab = " To get a GPU: click Runtime > Change runtime type > T4 GPU > Save, then click ▶️ on this cell again."
    return slow + (colab if in_colab else "")


# ---------------------------------------------------------------- plain helpers for the notebook cells
SHOW = 60  # words of a passage shown on screen: a whole chunk of up to 60 words, the start of a longer one


def _short(text: str) -> str:
    words = text.split()
    return " ".join(words[:SHOW]) + (" ..." if len(words) > SHOW else "")


def ready() -> bool:
    """True once the Setup cell has loaded the models; otherwise says to run it."""
    if MODELS is None:
        print(NOT_READY)
        return False
    return True


def write_answer(prompt: str) -> None:
    """Send the prompt to the AI model and print its answer as it is written."""
    for piece in MODELS.generator.generate(prompt):
        print(piece, end="", flush=True)
    print()


def show_passages(passages: list[str]) -> None:
    print("Passages sent to the AI:")
    for n, passage in enumerate(passages, start=1):
        print(f"{n}. {_short(passage)}")
    print()


def _bare(text: str) -> list[str]:
    """The words of a text, without the punctuation around them ("foot." and "foot" match)."""
    return [word.strip(".,;:!?\"'()“”‘’") for word in text.split()]


def where_is(chunks: list[str], phrase: str) -> None:
    """Say which chunks contain a phrase, or show where it was cut in two."""
    norm = " ".join(phrase.split()).lower()
    flat = [" ".join(c.split()).lower() for c in chunks]
    found = [i + 1 for i, c in enumerate(flat) if norm and norm in c]
    if found:
        label = "chunk" if len(found) == 1 else "chunks"
        print(f'"{phrase}" is in {label} {", ".join(map(str, found))}.')
        return
    words = _bare(norm)
    for i in range(len(flat) - 1):
        left, right = _bare(flat[i]), _bare(flat[i + 1])
        for j in range(len(words) - 1, 0, -1):
            if left[-j:] == words[:j] and right[: len(words) - j] == words[j:]:
                print(f'"{phrase}" was cut in two, between chunk {i + 1} and chunk {i + 2}.')
                print(f"Chunk {i + 1} ends with: ...{chunks[i][-80:]}")
                print(f"Chunk {i + 2} starts with: {chunks[i + 1][:80]}...")
                return
    print(f'"{phrase}" is not in any chunk.')


_ATTRIBUTION = ("Title:", "Source:", "Retrieved:", "License:", "Shortened:")
_HEADING = re.compile(r"^(={2,3})\s*(.+?)\s*\1\s*$")


def sections(document: str) -> list[str]:
    """One passage per section (== and === headings). The heading path is the first line."""
    lines = document.splitlines()
    head = []
    for line in lines:
        if not line.strip():
            break
        head.append(line)
    if head and all(line.startswith(_ATTRIBUTION) for line in head):
        lines = lines[len(head) :]

    passages: list[str] = []
    title = "Introduction"
    parent = ""
    body: list[str] = []

    def flush() -> None:
        text = "\n".join(body).strip()
        if text:
            passages.append(f"{title}\n{text}")
        body.clear()

    for line in lines:
        match = _HEADING.match(line.strip())
        if not match:
            body.append(line)
            continue
        flush()
        name = match.group(2)
        if len(match.group(1)) == 2:
            parent = name
            title = name
        else:
            title = f"{parent}: {name}" if parent else name
    flush()
    return passages


# ---------------------------------------------------------------- shown in the notebook, one section at a time
# Each function below is also shown, as it is here, in a collapsed "Code" cell of its section.

# Section 1
def ask(question: str) -> None:
    """Ask the AI model alone: no documents, only what it learned during its training."""
    if not ready():
        return
    print("Answer:")
    write_answer(question)  # the question is sent as it is, with nothing added


# Section 2
def embed(texts: list[str]) -> np.ndarray:
    """Turn each text into an embedding: a list of 384 numbers that captures its meaning."""
    return MODELS.embedder.embed(texts)


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """How close two embeddings point: 1 = same direction (same meaning), 0 = unrelated."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def score(a: str, b: str) -> float:
    """The similarity score of two texts: embed both, then compare the two lists of numbers."""
    if MODELS is None:
        print(NOT_READY)
        return 0.0
    numbers_a, numbers_b = embed([a, b])
    return cosine(numbers_a, numbers_b)


def similarity(a: str, b: str) -> None:
    """Print how similar two texts are, e.g. 'Similarity: 0.83'."""
    if not ready():
        return
    print(f"Similarity: {score(a, b):.2f}")


# Section 3
def top_passages(question: str, passages: list[str], top: int = 3) -> list[tuple[float, str]]:
    """The `top` passages closest in meaning to the question, best first, as (score, passage)."""
    if not passages:
        return []
    # 1. Embed the question and every passage (one call, so it is fast).
    question_numbers, *passage_numbers = embed([question] + list(passages))
    # 2. Score every passage against the question.
    scored = [(cosine(question_numbers, numbers), passage) for numbers, passage in zip(passage_numbers, passages)]
    # 3. Sort from the highest score to the lowest, and keep the first `top`.
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return scored[:top]


def search(question: str, passages: list[str], top: int = 3) -> None:
    """Print the passages closest to the question: rank, score, start of the text."""
    if not ready():
        return
    for n, (s, passage) in enumerate(top_passages(question, passages, top), start=1):
        print(f"{n}. {s:.2f}  {_short(passage)}")


# Section 4
def chunk_overlap(text: str, size: int, overlap: int) -> list[str]:
    """Cut text into pieces of `size` words; each piece repeats the last `overlap` words of the one before."""
    overlap = max(0, min(overlap, size - 1))  # an overlap as long as the chunk would never move forward
    step = size - overlap  # how far each new chunk starts after the previous one
    words = text.split()
    chunks = []
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start : start + size]))
        if start + size >= len(words):  # this chunk reached the last word
            break
    return chunks


def split(document: str, words_per_chunk: int = 60, overlap: int = 0) -> list[str]:
    """Cut a document into chunks of words. 0 words means one chunk for everything."""
    if words_per_chunk <= 0:
        chunks = [document.strip()] if document.strip() else []
    else:
        chunks = chunk_overlap(document, words_per_chunk, max(0, overlap))
    print(f"Cut into {len(chunks)} {'chunk' if len(chunks) == 1 else 'chunks'}.")
    return chunks


# Section 5
def build_prompt(question: str, contexts: list[str], allow_unknown: bool) -> str:
    """The exact text sent to the model: instructions, the retrieved passages, the question."""
    lines = ["Answer the question using the context below."]
    if allow_unknown:
        lines.append('If the answer is not in the context, say "I don\'t know based on these documents."')
    lines += ["", "Context:"]
    lines += [f"[{i}] {c}" for i, c in enumerate(contexts, start=1)]
    lines += ["", f"Question: {question}", "Answer:"]
    return "\n".join(lines)


def ask_with_documents(question: str, passages: list[str], top: int = 3, say_dont_know: bool = True) -> None:
    """RAG: find the passages closest to the question, then let the AI answer from them."""
    if not ready():
        return
    # 1. Retrieve: embed the question and every passage, keep the `top` closest (section 3).
    best = [passage for _, passage in top_passages(question, passages, top)]
    show_passages(best)
    # 2. Augment: paste those passages into the prompt, above the question.
    prompt = build_prompt(question, best, say_dont_know)
    # 3. Generate: the AI writes its answer from that prompt.
    print("Answer:")
    write_answer(prompt)


print("Loading the two AI models. This takes 1 to 2 minutes...")
MODELS = load_models()
print(status_line(MODELS.device))


## 1. The problem (3 min)

The notebook uses one document: the programme of this workshop, with its schedule, the AIAI Lab, the Ka Hawaiʻi Pae ʻĀina Pod and short bios of the speakers. The AI model in this notebook was trained long before the programme was written, so it has never read it.

**Do this:**
1. Run the cell below. It asks the AI when the hands-on session starts. The programme says 1:00 pm, on Friday.
2. Read the answer. Is it right?
3. Change the question to "Who gives the introduction to machine learning at the Future Imaginaries of Indigenous AI workshop?" and run the cell again. Keep the quotes.

In [ ]:
# Change the question and run the cell again.
ask("When does the hands-on session start at the Future Imaginaries of Indigenous AI workshop?")

In [ ]:
#@title 🔍 Code: how ask() works (optional: double-click to see the code) { display-mode: "form" }
# The code behind ask(), exactly as the Setup cell defines it.
# You don't need to run this cell: Setup already did. Running it again changes nothing.


def ask(question: str) -> None:
    """Ask the AI model alone: no documents, only what it learned during its training."""
    if not ready():
        return
    print("Answer:")
    write_answer(question)  # the question is sent as it is, with nothing added


The AI has never read this programme. It may say so, or it may make up an answer that sounds right.

## 2. Embeddings: comparing meanings (22 min)

To compare meanings, the computer turns each sentence into a list of 384 numbers, called an embedding. Sentences with similar meanings get similar lists. The similarity score compares two lists: close to 0 means unrelated, and the closer to 1, the closer the meanings.

**Do this:**
1. The cell below compares "We learn about machine learning." with three other sentences. Guess which one gets the highest score, then run the cell.
2. Look at the score of the second sentence. It shares the word "machine" with the first one, but its score is low.

In [ ]:
similarity("We learn about machine learning.", "Students study how AI is trained.")
similarity("We learn about machine learning.", "The washing machine is broken.")
similarity("We learn about machine learning.", "I had soup for lunch.")

In [ ]:
#@title 🔍 Code: how similarity() works (optional: double-click to see the code) { display-mode: "form" }
# The code behind embed(), cosine(), score(), similarity(), exactly as the Setup cell defines it.
# You don't need to run this cell: Setup already did. Running it again changes nothing.


def embed(texts: list[str]) -> np.ndarray:
    """Turn each text into an embedding: a list of 384 numbers that captures its meaning."""
    return MODELS.embedder.embed(texts)


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """How close two embeddings point: 1 = same direction (same meaning), 0 = unrelated."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def score(a: str, b: str) -> float:
    """The similarity score of two texts: embed both, then compare the two lists of numbers."""
    if MODELS is None:
        print(NOT_READY)
        return 0.0
    numbers_a, numbers_b = embed([a, b])
    return cosine(numbers_a, numbers_b)


def similarity(a: str, b: str) -> None:
    """Print how similar two texts are, e.g. 'Similarity: 0.83'."""
    if not ready():
        return
    print(f"Similarity: {score(a, b):.2f}")


**Do this:**
1. Guess the score of "I love winter." and "I hate winter.", then run the cell.
2. Write two sentences of your own between the quotes and run the cell again.

In [ ]:
# Change the two sentences and run the cell again.
similarity("I love winter.", "I hate winter.")

The score tells you whether two sentences talk about the same thing, not whether they agree.

## 3. Search: finding the right passage (15 min)

Search uses the same scores. It compares your question with every passage and lists the closest ones first, with their scores. Here a passage is one section of the programme. There are 20: the Introduction, About Abundant Intelligences, The Future Imaginaries workshops, Day 1, Day 2, The Ka Hawaiʻi Pae ʻĀina Pod, The AIAI Lab, and one per speaker.

**Do this:**
1. Guess which section will come first for "Who are the co-directors of the AIAI Lab?", then run the cell.

In [ ]:
passages = sections(PROGRAMME)
search("Who are the co-directors of the AIAI Lab?", passages)

In [ ]:
#@title 🔍 Code: how search() works (optional: double-click to see the code) { display-mode: "form" }
# The code behind top_passages(), search(), exactly as the Setup cell defines it.
# You don't need to run this cell: Setup already did. Running it again changes nothing.


def top_passages(question: str, passages: list[str], top: int = 3) -> list[tuple[float, str]]:
    """The `top` passages closest in meaning to the question, best first, as (score, passage)."""
    if not passages:
        return []
    # 1. Embed the question and every passage (one call, so it is fast).
    question_numbers, *passage_numbers = embed([question] + list(passages))
    # 2. Score every passage against the question.
    scored = [(cosine(question_numbers, numbers), passage) for numbers, passage in zip(passage_numbers, passages)]
    # 3. Sort from the highest score to the lowest, and keep the first `top`.
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return scored[:top]


def search(question: str, passages: list[str], top: int = 3) -> None:
    """Print the passages closest to the question: rank, score, start of the text."""
    if not ready():
        return
    for n, (s, passage) in enumerate(top_passages(question, passages, top), start=1):
        print(f"{n}. {s:.2f}  {_short(passage)}")


**Do this:**
1. Run the cell below. This is the question from section 1. The answer, Hugo Berard and Hamza Abdelhedi, is in the section "Day 2: Friday, November 20". Which section comes first?

In [ ]:
passages = sections(PROGRAMME)
search("Who gives the introduction to machine learning?", passages)

**Do this:**
1. Run the cell below. No section is about pizza. Look at the scores.
2. Change the question to one of your own and run the cell again.

In [ ]:
# Change the question and run the cell again.
passages = sections(PROGRAMME)
search("What is the best pizza topping?", passages)

Search always returns passages, even when none of them answers the question, and the best one is not always first.

## 4. Splitting: cutting the programme into chunks (15 min)

The embedding model reads only about the first 200 words of a passage and ignores the rest. Even those 200 words talk about many things, so the embedding is a mix of them. The names you looked for are one sentence of the Day 2 section, among many times, rooms and talks, so that section does not stand out for the question. The fix is to cut the programme into short pieces, called chunks, each with its own embedding.

**Do this:**
1. Run the cell below. It cuts the programme into chunks of 50 words and searches them. Are the names in the first result?
2. Change 50 to 5 and run the cell again. The first results are bits of sentences, and none has the names.
3. Change it to 0 and run the cell again. 0 means one chunk for the whole programme. Its score is lower: the model reads only the first 200 words of that chunk, and they talk about many things.

In [ ]:
# Change 50 to 5, then to 0, and run the cell again each time.
chunks = split(PROGRAMME, words_per_chunk=50)
search("Who gives the introduction to machine learning?", chunks)

In [ ]:
#@title 🔍 Code: how split() works (optional: double-click to see the code) { display-mode: "form" }
# The code behind chunk_overlap(), split(), exactly as the Setup cell defines it.
# You don't need to run this cell: Setup already did. Running it again changes nothing.


def chunk_overlap(text: str, size: int, overlap: int) -> list[str]:
    """Cut text into pieces of `size` words; each piece repeats the last `overlap` words of the one before."""
    overlap = max(0, min(overlap, size - 1))  # an overlap as long as the chunk would never move forward
    step = size - overlap  # how far each new chunk starts after the previous one
    words = text.split()
    chunks = []
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start : start + size]))
        if start + size >= len(words):  # this chunk reached the last word
            break
    return chunks


def split(document: str, words_per_chunk: int = 60, overlap: int = 0) -> list[str]:
    """Cut a document into chunks of words. 0 words means one chunk for everything."""
    if words_per_chunk <= 0:
        chunks = [document.strip()] if document.strip() else []
    else:
        chunks = chunk_overlap(document, words_per_chunk, max(0, overlap))
    print(f"Cut into {len(chunks)} {'chunk' if len(chunks) == 1 else 'chunks'}.")
    return chunks


A chunk can also end in the middle of an answer.

**Do this:**
1. Run the cell below. It looks for the sentence that says who gives the talk "Can we trust AI?". That sentence was cut in two: it starts at the end of one chunk and ends at the start of the next.

In [ ]:
chunks = split(PROGRAMME, words_per_chunk=50)
where_is(chunks, "Yorguin José Mantilla Ramos gives a 15-minute talk at 2:30 pm")

Overlap fixes this: each chunk starts with the last words of the chunk before it.

**Do this:**
1. Run the cell below. With an overlap of 10 words, the sentence is whole again, in one chunk.
2. Change 10 to 0 and run the cell again.

In [ ]:
# Change 10 to 0 and run the cell again.
chunks = split(PROGRAMME, words_per_chunk=50, overlap=10)
where_is(chunks, "Yorguin José Mantilla Ramos gives a 15-minute talk at 2:30 pm")

Long chunks mix many topics, and short chunks, or chunks cut in the wrong place, lose the answer.

## 5. Answering: the AI reads the chunks (15 min)

Now the AI gets the chunks that search finds. Each cell below cuts the programme as in section 4 (50 words, overlap 10), then sends the AI one message: an instruction, the 3 best chunks and your question. The instruction says to answer from the chunks, and to say "I don't know based on these documents." when the answer is not there.

**Do this:**
1. Run the cell below. It asks when the hands-on session starts again, this time with the programme. It first shows the chunks it sent, under "Passages sent to the AI".
2. Compare the answer with the one you got in section 1.

In [ ]:
chunks = split(PROGRAMME, words_per_chunk=50, overlap=10)
ask_with_documents("When does the hands-on session start?", chunks)

In [ ]:
#@title 🔍 Code: how ask_with_documents() works, the whole RAG (optional: double-click to see the code) { display-mode: "form" }
# The code behind build_prompt(), ask_with_documents(), exactly as the Setup cell defines it.
# You don't need to run this cell: Setup already did. Running it again changes nothing.


def build_prompt(question: str, contexts: list[str], allow_unknown: bool) -> str:
    """The exact text sent to the model: instructions, the retrieved passages, the question."""
    lines = ["Answer the question using the context below."]
    if allow_unknown:
        lines.append('If the answer is not in the context, say "I don\'t know based on these documents."')
    lines += ["", "Context:"]
    lines += [f"[{i}] {c}" for i, c in enumerate(contexts, start=1)]
    lines += ["", f"Question: {question}", "Answer:"]
    return "\n".join(lines)


def ask_with_documents(question: str, passages: list[str], top: int = 3, say_dont_know: bool = True) -> None:
    """RAG: find the passages closest to the question, then let the AI answer from them."""
    if not ready():
        return
    # 1. Retrieve: embed the question and every passage, keep the `top` closest (section 3).
    best = [passage for _, passage in top_passages(question, passages, top)]
    show_passages(best)
    # 2. Augment: paste those passages into the prompt, above the question.
    prompt = build_prompt(question, best, say_dont_know)
    # 3. Generate: the AI writes its answer from that prompt.
    print("Answer:")
    write_answer(prompt)


**Do this:**
1. Run the cell below. With top=1, the AI gets only the best chunk, and that chunk is about Friday, not Thursday.
2. Change top=1 to top=3 and run the cell again.

In [ ]:
# Change top=1 to top=3 and run the cell again.
chunks = split(PROGRAMME, words_per_chunk=50, overlap=10)
ask_with_documents("Which room is used on Thursday?", chunks, top=1)

**Do this:**
1. Run the cell below. The programme does not give the Wi-Fi password.
2. Change True to False and run the cell again. This removes the "I don't know" instruction. Compare the two answers.

In [ ]:
# Change True to False and run the cell again.
chunks = split(PROGRAMME, words_per_chunk=50, overlap=10)
ask_with_documents("What is the Wi-Fi password?", chunks, say_dont_know=True)

**Do this:**
1. Run the cell below. It asks who gives the talk "Can we trust AI?", with the overlap of 10 words from section 4.
2. Change 10 to 0 and run the cell again. The sentence is cut in two again. Look at the chunks sent to the AI: is the speaker's name still there?

In [ ]:
# Change 10 to 0 and run the cell again.
chunks = split(PROGRAMME, words_per_chunk=50, overlap=10)
ask_with_documents("Who gives the talk Can we trust AI?", chunks)

The AI can only answer as well as the chunks it gets.

## 6. Wrap-up (15 min)

RAG has four steps:
1. Cut the documents into chunks.
2. Compute an embedding for each chunk.
3. For each question, find the chunks closest to it.
4. Send those chunks and the question to the AI, which writes the answer.

**Do this:**
1. Run the cell below. It asks the question from sections 1 and 3 again. Its two lines do all four steps.
2. Change the question to one of your own about the workshop and run the cell again.

In [ ]:
# Change the question and run the cell again.
chunks = split(PROGRAMME, words_per_chunk=50, overlap=10)
ask_with_documents("Who gives the introduction to machine learning?", chunks)

Each choice can change the answers: the chunk size, the overlap, the number of chunks sent and the instruction.

### Running it on your own computer

In Colab, your text goes to Google's computers. On your own computer, your documents stay with you. Ollama, the app from Part 2, runs AI models on your own computer. This notebook also runs in Jupyter on a laptop.

**Do this:**
1. Click **File > Download > Download .ipynb** to save this notebook.
2. Install Python 3.11 or newer.
3. In a terminal, run `pip install sentence-transformers transformers torch accelerate jupyterlab`, then `jupyter lab`.
4. In Jupyter, open the file you saved.

### Sources

- The document: the programme of Future Imaginaries of Indigenous AI, a workshop by Abundant Intelligences, with facts from [abundant-intelligences.net](https://abundant-intelligences.net/ka-hawaii-pae-aina-pod/) and the speakers' own pages, retrieved 2026-11-20. The bio of Solomon Enos is from the Wikipedia article "Solomon Enos" ([CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/)).
- The models: all-MiniLM-L6-v2 for the embeddings, and Qwen3-1.7B (with a GPU) or Qwen3-0.6B (without one) for the answers.

## Bonus: your own text file

You can ask questions about your own text. In Colab, the file you upload goes to Google's computers, so only upload text you are allowed to share.

**Do this:**
1. Save your document as a plain text file. In Google Docs, click **File > Download > Plain text (.txt)**. In Word, click **File > Save As** and choose **Plain Text**.
2. Run the first cell below. Click **Choose Files** under the cell (in some browsers it says **Browse**) and pick your file.
3. In the second cell, replace the question with your own. Keep the quotes. Then run the cell.

In [ ]:
from google.colab import files
uploaded = files.upload()
my_text = list(uploaded.values())[0].decode("utf-8", errors="replace")

In [ ]:
# Change the question and run the cell again.
my_chunks = split(my_text, words_per_chunk=50, overlap=10)
ask_with_documents("Type your question here", my_chunks)

Not in Colab? Put your file next to the notebook and replace the first cell with this line, with your file's name:

```
my_text = open("my_file.txt", encoding="utf-8").read()
```

The same four steps work on any text, though this embedding model searches English text best.